# Przepływ pracy Human-in-the-Loop z Microsoft Agent Framework

## 🎯 Cele nauki

W tym notatniku nauczysz się, jak wdrożyć przepływy pracy **human-in-the-loop** za pomocą `RequestInfoExecutor` z Microsoft Agent Framework. Ten potężny wzorzec pozwala na wstrzymanie przepływu AI w celu zebrania informacji od człowieka, dzięki czemu twoi agenci stają się interaktywni i dają ludziom kontrolę nad kluczowymi decyzjami.

## 🔄 Czym jest Human-in-the-Loop?

**Human-in-the-loop (HITL)** to wzorzec projektowy, w którym agenci AI wstrzymują wykonanie, aby poprosić o dane od człowieka przed kontynuacją. Jest to niezbędne dla:

- ✅ **Decyzji krytycznych** - Uzyskanie zatwierdzenia od człowieka przed podjęciem ważnych działań
- ✅ **Sytuacji niejednoznacznych** - Pozwolenie ludziom na wyjaśnienie, gdy AI jest niepewne
- ✅ **Preferencji użytkownika** - Zapytanie użytkowników o wybór spośród kilku opcji
- ✅ **Zgodności i bezpieczeństwa** - Zapewnienie nadzoru ludzkiego dla operacji regulowanych
- ✅ **Interaktywnych doświadczeń** - Budowanie agentów konwersacyjnych reagujących na dane od użytkownika

## 🏗️ Jak to działa w Microsoft Agent Framework

Framework oferuje trzy kluczowe komponenty dla HITL:

1. **`RequestInfoExecutor`** - Specjalny executor, który wstrzymuje przepływ pracy i generuje `RequestInfoEvent`
2. **`RequestInfoMessage`** - Klasa bazowa dla typowanych ładunków żądania wysyłanych do ludzi
3. **`RequestResponse`** - Koreluje odpowiedzi ludzi z oryginalnymi prośbami za pomocą `request_id`

**Wzorzec przepływu pracy:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Nasz przykład: Rezerwacja hotelu z potwierdzeniem użytkownika

Rozbudujemy warunkowy przepływ pracy, dodając potwierdzenie człowieka **przed** zasugerowaniem alternatywnych destynacji:

1. Użytkownik prosi o destynację (np. "Paryż")
2. `availability_agent` sprawdza dostępność pokoi
3. **Jeśli brak pokoi** → `confirmation_agent` pyta "Czy chcesz zobaczyć alternatywy?"
4. Przepływ pracy **wstrzymuje się** za pomocą `RequestInfoExecutor`
5. **Człowiek odpowiada** "tak" lub "nie" przez wejście w konsoli
6. `decision_manager` kieruje na podstawie odpowiedzi:
   - **Tak** → Pokaż alternatywne destynacje
   - **Nie** → Anuluj żądanie rezerwacji
7. Wyświetl ostateczny wynik

To pokazuje, jak dać użytkownikom kontrolę nad sugestiami agenta!

---

Zaczynajmy! 🚀


## Krok 1: Importowanie Wymaganych Bibliotek

Importujemy standardowe komponenty Frameworka Agentów oraz **klasy specyficzne dla „człowiek w pętli”**:
- `RequestInfoExecutor` - Executor, który zatrzymuje przepływ pracy na dane wejściowe od człowieka
- `RequestInfoEvent` - Zdarzenie emitowane, gdy wymagane jest dane wejściowe od człowieka
- `RequestInfoMessage` - Klasa bazowa dla typowanych ładunków zapytań
- `RequestResponse` - Łączy odpowiedzi człowieka z zapytaniami
- `WorkflowOutputEvent` - Zdarzenie do wykrywania wyników przepływu pracy


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## Krok 2: Zdefiniuj modele Pydantic dla ustrukturyzowanych wyników

Modele te definiują **schemat**, który będą zwracać agenci. Zachowujemy wszystkie modele z warunkowego przepływu pracy i dodajemy:

**Nowe dla Human-in-the-Loop:**
- `HumanFeedbackRequest` - podklasa `RequestInfoMessage`, która definiuje ładunek żądania wysyłanego do ludzi
  - Zawiera `prompt` (pytanie do zadania) oraz `destination` (kontekst dotyczący niedostępnego miasta)


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## Krok 3: Utwórz narzędzie do rezerwacji hotelu

To samo narzędzie co w warunkowym przepływie pracy – sprawdza, czy dostępne są pokoje w miejscu docelowym.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## Krok 4: Zdefiniuj funkcje warunkowe do trasowania

Potrzebujemy **czterech funkcji warunkowych** dla naszego workflow z człowiekiem w pętli:

**Z warunkowego workflow:**
1. `has_availability_condition` - Trasuje, gdy hotele SĄ dostępne
2. `no_availability_condition` - Trasuje, gdy hotele NIE są dostępne

**Nowe dla człowieka w pętli:**
3. `user_wants_alternatives_condition` - Trasuje, gdy użytkownik mówi "tak" dla alternatyw
4. `user_declines_alternatives_condition` - Trasuje, gdy użytkownik mówi "nie" dla alternatyw


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## Krok 5: Utwórz Executor Menedżera Decyzji

To jest **sedno wzorca człowiek-w-pętli**! `DecisionManager` to niestandardowy `Executor`, który:

1. **Odbiera opinię od człowieka** za pomocą obiektów `RequestResponse`
2. **Przetwarza decyzję użytkownika** (tak/nie)
3. **Kieruje przepływem pracy** wysyłając wiadomości do odpowiednich agentów

Kluczowe cechy:
- Używa dekoratora `@handler`, aby udostępnić metody jako kroki przepływu pracy
- Odbiera `RequestResponse[HumanFeedbackRequest, str]` zawierający zarówno oryginalne żądanie, jak i odpowiedź użytkownika
- Zwraca proste wiadomości "tak" lub "nie", które wywołują nasze funkcje warunkowe


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## Krok 6: Utwórz niestandardowy executor wyświetlania

Ten sam executor wyświetlania z warunkowego przepływu pracy - zwraca ostateczne wyniki jako wynik przepływu pracy.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## Krok 7: Załaduj zmienne środowiskowe

Skonfiguruj klienta LLM (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## Krok 8: Utwórz agentów AI i wykonawców

Tworzymy **sześć komponentów przepływu pracy**:

**Agenci (opakowani w AgentExecutor):**
1. **availability_agent** - Sprawdza dostępność hotelu za pomocą narzędzia
2. **confirmation_agent** - 🆕 Przygotowuje prośbę o potwierdzenie od człowieka
3. **alternative_agent** - Sugeruje alternatywne miasta (gdy użytkownik mówi tak)
4. **booking_agent** - Zachęca do rezerwacji (gdy są dostępne pokoje)
5. **cancellation_agent** - 🆕 Obsługuje wiadomość o anulowaniu (gdy użytkownik mówi nie)

**Specjalni wykonawcy:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor`, który wstrzymuje przepływ pracy na czas wprowadzenia danych przez człowieka
7. **decision_manager** - 🆕 Niestandardowy wykonawca, który kieruje na podstawie odpowiedzi człowieka (już zdefiniowany powyżej)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## Krok 9: Budowanie przepływu pracy z udziałem człowieka w pętli

Teraz tworzymy wykres przepływu pracy z **warunkowym kierowaniem** włącznie ze ścieżką z udziałem człowieka w pętli:

**Struktura przepływu pracy:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Kluczowe krawędzie:**
- `availability_agent → confirmation_agent` (gdy brak pokoi)
- `confirmation_agent → prepare_human_request` (transformacja typu)
- `prepare_human_request → request_info_executor` (pauza na człowieka)
- `request_info_executor → decision_manager` (zawsze - dostarcza RequestResponse)
- `decision_manager → alternative_agent` (gdy użytkownik mówi "tak")
- `decision_manager → cancellation_agent` (gdy użytkownik mówi "nie")
- `availability_agent → booking_agent` (gdy pokoje są dostępne)
- Wszystkie ścieżki kończą się na `display_result`


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## Krok 10: Uruchom Test Case 1 - Miasto BEZ dostępności (Paryż z potwierdzeniem człowieka)

Ten test demonstruje **pełny cykl z udziałem człowieka**:

1. Zapytanie o hotel w Paryżu
2. availability_agent sprawdza → Brak pokoi
3. confirmation_agent tworzy pytanie dla człowieka
4. request_info_executor **wstrzymuje przepływ pracy** i emituje `RequestInfoEvent`
5. **Aplikacja wykrywa zdarzenie i pyta użytkownika w konsoli**
6. Użytkownik wpisuje "tak" lub "nie"
7. Aplikacja wysyła odpowiedź za pomocą `send_responses_streaming()`
8. decision_manager kieruje dalej na podstawie odpowiedzi
9. Wyświetlany jest ostateczny wynik

**Kluczowy wzorzec:**
- Używaj `workflow.run_stream()` przy pierwszej iteracji
- Używaj `workflow.send_responses_streaming(pending_responses)` przy kolejnych iteracjach
- Nasłuchuj `RequestInfoEvent`, aby wykryć gdy potrzebny jest wkład człowieka
- Nasłuchuj `WorkflowOutputEvent`, aby przechwycić ostateczne wyniki


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## Krok 11: Uruchom test Case 2 - Miasto Z dostępnością (Sztokholm - Bez udziału człowieka)

Ten test demonstruje **bezpośrednią ścieżkę**, gdy pokoje są dostępne:

1. Prośba o hotel w Sztokholmie
2. availability_agent sprawdza → Pokoje dostępne ✅
3. booking_agent sugeruje rezerwację
4. display_result pokazuje potwierdzenie
5. **Brak potrzeby udziału człowieka!**

Przepływ pracy całkowicie omija ścieżkę z udziałem człowieka, gdy dostępne są pokoje.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Kluczowe wnioski i najlepsze praktyki human-in-the-loop

### ✅ Czego się nauczyłeś:

#### 1. **Wzorzec RequestInfoExecutor**
Wzorzec human-in-the-loop w Microsoft Agent Framework wykorzystuje trzy kluczowe komponenty:
- `RequestInfoExecutor` - Wstrzymuje przepływ pracy i emituje zdarzenia
- `RequestInfoMessage` - Klasa bazowa dla typowanych ładunków żądań (stwórz podklasę!)
- `RequestResponse` - Koreluje odpowiedzi ludzi z oryginalnymi żądaniami

**Kluczowe zrozumienie:**
- `RequestInfoExecutor` NIE zbiera danych wejściowych samodzielnie - tylko wstrzymuje przepływ pracy
- Twój kod aplikacji musi nasłuchiwać `RequestInfoEvent` i zbierać dane wejściowe
- Musisz wywołać `send_responses_streaming()` z słownikiem mapującym `request_id` do odpowiedzi użytkownika

#### 2. **Wzorzec wykonywania strumieniowego**
```python
# Pierwsza iteracja
stream = workflow.run_stream(initial_request)

# Kolejne iteracje (po wprowadzeniu przez człowieka)
stream = workflow.send_responses_streaming(pending_responses)

# Zawsze przetwarzaj zdarzenia
events = [event async for event in stream]
```

#### 3. **Architektura zdarzeniowa**
Nasłuchuj konkretnych zdarzeń, aby kontrolować przepływ pracy:
- `RequestInfoEvent` - Wymagane dane wejściowe od człowieka (przepływ pracy wstrzymany)
- `WorkflowOutputEvent` - Dostępny jest ostateczny wynik (przepływ pracy zakończony)
- `WorkflowStatusEvent` - Zmiany stanu (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS itp.)

#### 4. **Niestandardowi wykonawcy z @handler**
`DecisionManager` pokazuje, jak tworzyć wykonawców, którzy:
- Używają dekoratora `@handler`, by udostępniać metody jako kroki przepływu pracy
- Odbierają typowane wiadomości (np. `RequestResponse[HumanFeedbackRequest, str]`)
- Kierują przepływem pracy, wysyłając wiadomości do innych wykonawców
- Mają dostęp do kontekstu przez `WorkflowContext`

#### 5. **Warunkowe kierowanie z decyzjami ludzkimi**
Możesz tworzyć funkcje warunkowe oceniające odpowiedzi ludzi:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Praktyczne zastosowania:

1. **Przepływy pracy zatwierdzania**
   - Uzyskaj zatwierdzenie od menedżera przed przetwarzaniem raportów wydatków
   - Wymagaj przeglądu ludzkiego przed wysłaniem zautomatyzowanych e-maili
   - Potwierdź transakcje o wysokiej wartości przed wykonaniem

2. **Moderacja treści**
   - Oznacz podejrzane treści do przeglądu przez człowieka
   - Poproś moderatorów o podjęcie ostatecznej decyzji w trudnych przypadkach
   - Eskaluj do ludzi, gdy pewność AI jest niska

3. **Obsługa klienta**
   - Pozwól AI automatycznie obsługiwać rutynowe pytania
   - Eskaluj złożone problemy do agentów ludzkich
   - Zapytaj klienta, czy chce rozmawiać z człowiekiem

4. **Przetwarzanie danych**
   - Poproś ludzi o rozstrzygnięcie niejednoznacznych wpisów danych
   - Potwierdź interpretacje AI dotyczące niejasnych dokumentów
   - Pozwól użytkownikom wybierać spośród różnych prawidłowych interpretacji

5. **Systemy krytyczne dla bezpieczeństwa**
   - Wymagaj potwierdzenia ludzkiego przed działaniami nieodwracalnymi
   - Uzyskaj zatwierdzenie przed dostępem do danych wrażliwych
   - Potwierdź decyzje w branżach regulowanych (opiece zdrowotnej, finansach)

6. **Interaktywne agentury**
   - Buduj boty konwersacyjne, które zadają pytania uzupełniające
   - Twórz kreatory prowadzące użytkowników przez skomplikowane procesy
   - Projektuj agentów współpracujących krok po kroku z ludźmi

### 🔄 Porównanie: z i bez human-in-the-loop

| Funkcja | Przepływ warunkowy | Przepływ human-in-the-loop |
|---------|---------------------|---------------------------|
| **Wykonanie** | Pojedyncze `workflow.run()` | Pętla z `run_stream()` + `send_responses_streaming()` |
| **Dane wejściowe użytkownika** | Brak (w pełni zautomatyzowane) | Interaktywne monity przez `input()` lub UI |
| **Komponenty** | Agenci + Wykonawcy | + RequestInfoExecutor + DecisionManager |
| **Zdarzenia** | Tylko AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent itd. |
| **Wstrzymywanie** | Brak wstrzymywania | Przepływ wstrzymany na RequestInfoExecutor |
| **Kontrola ludzka** | Brak kontroli ludzkiej | Ludzie podejmują kluczowe decyzje |
| **Przypadek użycia** | Automatyzacja | Współpraca i nadzór |

### 🚀 Zaawansowane wzorce:

#### Wiele punktów decyzji ludzkiej
Możesz mieć wiele węzłów `RequestInfoExecutor` w tym samym przepływie pracy:
```python
.add_edge(agent1, request_info_1)  # Pierwsza ludzka decyzja
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Druga ludzka decyzja
.add_edge(decision_manager_2, final_agent)
```

#### Obsługa limitów czasu
Wdrażaj limity czasu na odpowiedzi ludzkie:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Domyślnie bezpieczna opcja
```

#### Bogata integracja UI
Zamiast `input()`, integruj z UI webowym, Slack, Teams itp.:
```python
if isinstance(event, RequestInfoEvent):
    # Wyślij powiadomienie na preferowany kanał użytkownika
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Warunkowy human-in-the-loop
Proś o dane wejściowe od ludzi tylko w konkretnych sytuacjach:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Kieruj do człowieka tylko, jeśli pewność jest niska lub wartość wysoka
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Najlepsze praktyki:

1. **Zawsze dziedzicz po RequestInfoMessage**
   - Zapewnia bezpieczeństwo typów i walidację
   - Umożliwia bogaty kontekst dla renderowania UI
   - Jasno wskazuje cel każdego typu żądania

2. **Używaj opisowych monitów**
   - Dołącz kontekst, o co pytasz
   - Wyjaśnij konsekwencje każdego wyboru
   - Zachowuj pytania proste i jasne

3. **Obsługuj nieoczekiwane dane wejściowe**
   - Waliduj odpowiedzi użytkowników
   - Zapewnij wartości domyślne dla nieprawidłowych danych
   - Daj jasne komunikaty o błędach

4. **Śledź ID żądań**
   - Wykorzystaj korelację między request_id a odpowiedziami
   - Nie próbuj zarządzać stanem ręcznie

5. **Projektuj bez blokowania**
   - Nie blokuj wątków, czekając na dane wejściowe
   - Stosuj asynchroniczne wzorce w całym kodzie
   - Wspieraj równoczesne instancje przepływu pracy

### 📚 Powiązane koncepcje:

- **Agent Middleware** - Przechwytywanie wywołań agentów (poprzedni podręcznik)
- **Zarządzanie stanem przepływu pracy** - Zapisywanie stanu przepływu między uruchomieniami
- **Współpraca wielu agentów** - Łączenie human-in-the-loop z zespołami agentów
- **Architektury zdarzeniowe** - Buduj reaktywne systemy ze zdarzeniami

---

### 🎓 Gratulacje!

Opanowałeś przepływy human-in-the-loop z Microsoft Agent Framework! Teraz wiesz jak:
- ✅ Wstrzymywać przepływy pracy, by zebrać dane wejściowe od ludzi
- ✅ Używać RequestInfoExecutor i RequestInfoMessage
- ✅ Obsługiwać wykonywanie strumieniowe ze zdarzeniami
- ✅ Tworzyć niestandardowych wykonawców z @handler
- ✅ Kierować przepływami na podstawie decyzji ludzkich
- ✅ Budować interaktywne agentury AI współpracujące z ludźmi

**To kluczowy wzorzec do budowy godnych zaufania, kontrolowanych systemów AI!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
